# ET-aware percentile normalization

**Hypothesis.** T1 and T1ce carry the same anatomy at roughly the same intensity distribution,
except that the enhancing tumor adds a high-intensity tail to T1ce. So a single percentile divisor
applied to both is mis-set for T1ce: its 99th percentile sits *inside* the tumor tail, so dividing
by it shrinks T1ce relative to T1 and leaves a residual everywhere, not just at the tumor.

**Proposal.** Normalize FLAIR / T1 / T2 by their 99th percentile over the brain, but normalize T1ce
by the $(99 - 100 f_{ET})$-th percentile, where $f_{ET} = |ET| / |\mathrm{brain}|$ -- the idea being
that the ET voxels are exactly the tail you want to step over.

**Test.** If the hypothesis holds, after normalization $\Delta = \mathrm{T1ce} - \mathrm{T1}$ should
be near 0 across non-tumor brain and clearly positive inside ET.

Four schemes are compared:

| scheme | T1ce divisor | why it is here |
|---|---|---|
| `zscore` | (mean, std) | what `preprocessing/cmap.py` does today -- the baseline |
| `p99` | 99th pct of brain | one percentile for everything -- the thing the proposal fixes |
| `proposed` | $(99 - 100 f_{ET})$-th pct of brain | your rule |
| `oracle` | 99th pct of **brain minus ET** | the quantity `proposed` approximates |

`oracle` is the important control: it is what "the 99th percentile of healthy tissue" actually is, so
the gap between it and `proposed` tells you whether $99 - 100 f_{ET}$ is a good approximation or just
happens to help. Percentiles are taken **per volume over brain voxels**, matching `normalize_masked`
in the existing pipeline.

In [ ]:
import os, glob
import numpy as np
import h5py
import matplotlib.pyplot as plt
%matplotlib inline

# os.chdir('/scratch/ee2178/ImMAP')   # <-- EDIT to your repo root if needed

ROOT = "/home/ee2178/scratch/ee2178/datasets/BraTS/BraTS2021_DataSet_train"   # <-- EDIT
CONTRASTS = ["flair", "t1", "t1ce", "t2"]      # channel order in the h5 (cmap.yaml: contrasts)
FLAIR, T1, T1CE, T2 = 0, 1, 2, 3
P_BASE  = 99.0        # base percentile for the "healthy tissue" top
P_FLOOR = 50.0        # never let the ET correction drag the percentile below this
SEED    = 0

subjects = sorted(p for d in sorted(glob.glob(os.path.join(ROOT, "*")))
                  if os.path.isdir(d) for p in glob.glob(os.path.join(d, "*_img.h5")))
if not subjects:
    raise RuntimeError(f"no *_img.h5 under {ROOT}")


def has_keys(p, keys=("img_raw", "mask", "et")):
    with h5py.File(p, "r") as h:
        return all(k in h for k in keys)


usable = [p for p in subjects if has_keys(p)]
print(f"{len(subjects)} subject(s); {len(usable)} with img_raw + mask + et")
if not usable:
    raise RuntimeError(
        "need 'img_raw' (cmap.yaml save_raw_image: true) and 'et' (save_seg: true). The stored "
        "'img' is already z-scored, so a new normalization cannot be tested on it. Backfill ET "
        "with: python preprocessing/cmap.py --config config/BraTS/cmap.yaml --add-seg-only")
if len(usable) < len(subjects):
    print(f"[warn] {len(subjects) - len(usable)} subject(s) lack img_raw and/or et -- excluded")

## Load one subject

`img_raw` is unnormalized, unclipped, background already zeroed -- so every statistic below is taken
over the brain mask, never the whole slice (which is mostly zeros).

In [ ]:
def load_subject(path):
    """-> raw (n,H,W,4) float32 unnormalized, brain (n,H,W) bool, et (n,H,W) bool."""
    with h5py.File(path, "r") as h:
        raw = np.asarray(h["img_raw"]).astype(np.float32)
        brain = np.asarray(h["mask"])[..., 0] > 0.5
        et = np.asarray(h["et"])
        et = (et[..., 0] if et.ndim == 4 else et) > 0.5
    return raw, brain, et


rng = np.random.default_rng(SEED)
SUBJECT = usable[int(rng.integers(len(usable)))]      # <-- or paste a path here to pin one
raw, brain, et = load_subject(SUBJECT)

f_et = et.sum() / max(brain.sum(), 1)
print(f"subject : {os.path.basename(os.path.dirname(SUBJECT))}")
print(f"volume  : {raw.shape[0]} slices of {raw.shape[1]}x{raw.shape[2]}")
print(f"brain   : {brain.sum():,} voxels    ET: {et.sum():,} ({f_et:.3%} of brain)")
print(f"-> proposed T1ce percentile = {P_BASE} - 100*{f_et:.5f} = {P_BASE - 100 * f_et:.3f}")
print()
print(f"{'contrast':<8} {'min':>10} {'p50':>10} {'p99':>10} {'max':>10}   (over brain)")
for c, nm in enumerate(CONTRASTS):
    v = raw[..., c][brain]
    print(f"{nm:<8} {v.min():>10.1f} {np.percentile(v, 50):>10.1f} "
          f"{np.percentile(v, 99):>10.1f} {v.max():>10.1f}")

## The four schemes

In [ ]:
def pctl(vol_c, region, p):
    v = vol_c[region]
    return float(np.percentile(v, np.clip(p, 0.0, 100.0))) if v.size else 1.0


def normalize(raw, brain, et, scheme, p_base=P_BASE):
    """-> (normalized (n,H,W,4), info). Background stays 0."""
    out = np.zeros_like(raw)
    info = {"scheme": scheme, "pct_t1ce": None, "divisor": {}}
    if scheme == "zscore":                       # what cmap.py does today
        for c in range(4):
            v = raw[..., c][brain]
            out[..., c] = (raw[..., c] - v.mean()) / max(v.std(), 1e-8)
            info["divisor"][CONTRASTS[c]] = float(v.std())
    else:
        p_t1ce = p_base
        if scheme == "proposed":
            p_t1ce = max(p_base - 100.0 * (et.sum() / max(brain.sum(), 1)), P_FLOOR)
        for c in range(4):
            if scheme == "oracle" and c == T1CE:
                d = pctl(raw[..., c], brain & ~et, p_base)     # healthy tissue only
            else:
                d = pctl(raw[..., c], brain, p_t1ce if c == T1CE else p_base)
            info["divisor"][CONTRASTS[c]] = d
            out[..., c] = raw[..., c] / max(d, 1e-8)
        info["pct_t1ce"] = p_base if scheme == "oracle" else p_t1ce
    out *= brain[..., None]
    return out, info


SCHEMES = ["zscore", "p99", "proposed", "oracle"]
norm = {s: normalize(raw, brain, et, s) for s in SCHEMES}

print(f"{'scheme':<10} {'T1 div':>10} {'T1ce div':>10} {'ratio':>8} {'T1ce pct':>10}")
for s in SCHEMES:
    d, p = norm[s][1]["divisor"], norm[s][1]["pct_t1ce"]
    pct = "-" if p is None else f"{p:.3f}"
    print(f"{s:<10} {d['t1']:>10.2f} {d['t1ce']:>10.2f} "
          f"{d['t1ce'] / max(d['t1'], 1e-8):>8.3f} {pct:>10}")
print()
print("If the hypothesis holds, `proposed` should land close to `oracle` -- that is the whole")
print("claim: 99 - 100*f_ET is a usable stand-in for the 99th percentile of non-tumor tissue.")

## Does $\Delta = $ T1ce $-$ T1 isolate the tumor?

The figure of merit. Over non-ET brain we want $\Delta$ **centred on zero** (no global offset) and
**tight** (no anatomy leaking in). Inside ET we want it large. `separation` is the ET median divided
by the non-ET RMS -- how far the tumor stands out of the background residual. `leak` is the fraction
of healthy tissue that is at least half as bright as the tumor median.

In [ ]:
def residual_stats(x, brain, et):
    """x: normalized (n,H,W,4). -> the numbers the hypothesis makes predictions about."""
    d = x[..., T1CE] - x[..., T1]
    bg, tu = d[brain & ~et], d[et]
    s = {"bias": float(np.median(bg)),                  # want ~0
         "bg_rms": float(np.sqrt(np.mean(bg ** 2))),    # want small
         "bg_iqr": float(np.percentile(bg, 75) - np.percentile(bg, 25)),
         "et_med": float(np.median(tu)) if tu.size else float("nan")}
    s["separation"] = s["et_med"] / max(s["bg_rms"], 1e-8)
    s["leak"] = float(np.mean(bg > 0.5 * s["et_med"])) if tu.size else float("nan")
    return s


rows = {s: residual_stats(norm[s][0], brain, et) for s in SCHEMES}
print(f"{'scheme':<10} {'bias':>9} {'bg_rms':>8} {'bg_iqr':>8} {'et_med':>8} "
      f"{'separation':>11} {'leak':>8}")
for s in SCHEMES:
    r = rows[s]
    print(f"{s:<10} {r['bias']:>+9.4f} {r['bg_rms']:>8.4f} {r['bg_iqr']:>8.4f} "
          f"{r['et_med']:>8.4f} {r['separation']:>11.3f} {r['leak']:>8.2%}")
print()
print("bias -> 0 and separation large = the residual really is just the tumor.")

## Images

The slice with the most ET. Bottom row is $\Delta$ on a symmetric diverging scale (white = 0) with
the ET boundary drawn on -- under the hypothesis it should be flat white outside the contour.

In [ ]:
z = int(np.argmax(et.sum(axis=(1, 2))))
print(f"slice {z}: ET = {et[z].sum():,} px ({et[z].sum() / max(brain[z].sum(), 1):.2%} of brain)")

ncol = len(SCHEMES) + 1
fig, ax = plt.subplots(2, ncol, figsize=(3.0 * ncol, 6.2))
ax[0][0].imshow(raw[z, ..., T1] * brain[z], cmap="gray")
ax[0][0].set_title("T1 (raw)", fontsize=9)
ax[1][0].imshow(raw[z, ..., T1CE] * brain[z], cmap="gray")
ax[1][0].set_title("T1ce (raw)", fontsize=9)
for a in (ax[0][0], ax[1][0]):
    a.contour(et[z], levels=[0.5], colors="r", linewidths=0.6)

for j, s in enumerate(SCHEMES, start=1):
    x = norm[s][0]
    lo, hi = np.percentile(x[z, ..., T1CE][brain[z]], [1, 99])
    ax[0][j].imshow(x[z, ..., T1CE], cmap="gray", vmin=lo, vmax=hi)
    ax[0][j].set_title(f"{s}\nT1ce normalized", fontsize=9)
    d = (x[z, ..., T1CE] - x[z, ..., T1]) * brain[z]
    v = float(np.percentile(np.abs(d[brain[z]]), 99)) or 1.0
    im = ax[1][j].imshow(d, cmap="bwr", vmin=-v, vmax=v)
    ax[1][j].contour(et[z], levels=[0.5], colors="k", linewidths=0.6)
    ax[1][j].set_title(f"$\\Delta$   sep={rows[s]['separation']:.2f}", fontsize=9)
    plt.colorbar(im, ax=ax[1][j], fraction=0.046)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Histograms

Left: T1 vs T1ce over the brain -- the hypothesis says these should overlap once T1ce's tumor tail is
stepped over. Right: the residual, split into healthy tissue and ET. Under the hypothesis the healthy
curve is a narrow spike at 0 and the ET curve sits clearly to its right.

In [ ]:
fig, ax = plt.subplots(2, len(SCHEMES), figsize=(3.3 * len(SCHEMES), 6.0))
for j, s in enumerate(SCHEMES):
    x = norm[s][0]
    t1, t1ce = x[..., T1][brain], x[..., T1CE][brain]
    lo, hi = np.percentile(np.concatenate([t1, t1ce]), [0.5, 99.5])
    bins = np.linspace(lo, hi, 120)
    ax[0][j].hist(t1, bins=bins, histtype="step", density=True, label="T1")
    ax[0][j].hist(t1ce, bins=bins, histtype="step", density=True, label="T1ce")
    ax[0][j].set_title(s, fontsize=10); ax[0][j].set_yscale("log")
    if j == 0:
        ax[0][j].legend(fontsize=8); ax[0][j].set_ylabel("brain, log density")

    d = x[..., T1CE] - x[..., T1]
    bg, tu = d[brain & ~et], d[et]
    b2 = np.linspace(*np.percentile(d[brain], [0.5, 99.5]), 120)
    ax[1][j].hist(bg, bins=b2, histtype="step", density=True, label="healthy")
    if tu.size:
        ax[1][j].hist(tu, bins=b2, histtype="step", density=True, label="ET")
    ax[1][j].axvline(0, color="k", lw=0.8); ax[1][j].set_yscale("log")
    ax[1][j].set_xlabel(r"$\Delta$ = T1ce - T1")
    if j == 0:
        ax[1][j].legend(fontsize=8); ax[1][j].set_ylabel("log density")
plt.tight_layout(); plt.show()

## Robustness across subjects

One subject proves nothing -- $f_{ET}$ varies a lot, and so does how well $99 - 100 f_{ET}$ tracks
the oracle. This resamples `N_SUBJECTS` at random and reports the spread.

In [ ]:
N_SUBJECTS = 20

rng = np.random.default_rng(SEED)
pick = rng.choice(len(usable), size=min(N_SUBJECTS, len(usable)), replace=False)

recs = []
for i, si in enumerate(pick):
    p = usable[int(si)]
    try:
        r_, b_, e_ = load_subject(p)
    except Exception as err:
        print(f"[skip] {os.path.basename(p)}: {type(err).__name__}")
        continue
    if e_.sum() == 0:
        print(f"[skip] {os.path.basename(os.path.dirname(p))}: no ET voxels")
        continue
    rec = {"subject": os.path.basename(os.path.dirname(p)),
           "f_et": float(e_.sum() / max(b_.sum(), 1)), "div": {}}
    for s in SCHEMES:
        x_, i_ = normalize(r_, b_, e_, s)
        rec[s] = residual_stats(x_, b_, e_)
        rec["div"][s] = i_["divisor"]["t1ce"]
    recs.append(rec)
    print(f"  {i + 1}/{len(pick)} {rec['subject']}  f_ET={rec['f_et']:.3%}          ", end="\r")

print(f"\n\n{len(recs)} subject(s) with ET\n")
print(f"{'scheme':<10} {'|bias| med (IQR)':>24} {'bg_rms med (IQR)':>24} "
      f"{'separation med (IQR)':>26} {'leak med':>10}")
for s in SCHEMES:
    q = lambda k: np.percentile([abs(r[s][k]) if k == "bias" else r[s][k] for r in recs],
                                [25, 50, 75])
    b, g, sep, lk = q("bias"), q("bg_rms"), q("separation"), q("leak")
    print(f"{s:<10} {b[1]:>10.4f} ({b[0]:.3f}-{b[2]:.3f}) {g[1]:>10.4f} ({g[0]:.3f}-{g[2]:.3f}) "
          f"{sep[1]:>12.2f} ({sep[0]:.2f}-{sep[2]:.2f}) {lk[1]:>9.2%}")

# how closely does the proposed percentile track the oracle divisor? THE claim under test.
err = np.array([r["div"]["proposed"] / max(r["div"]["oracle"], 1e-8) - 1.0 for r in recs])
print(f"\nproposed / oracle T1ce divisor: median {np.median(err):+.2%}, "
      f"IQR [{np.percentile(err, 25):+.2%}, {np.percentile(err, 75):+.2%}], "
      f"|err| < 5% for {np.mean(np.abs(err) < 0.05):.0%} of subjects")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
xs = np.arange(len(SCHEMES))
for a, key, ttl in zip(ax, ["separation", "bg_rms", "bias"],
                       ["separation (higher = tumor stands out)",
                        "non-ET RMS (lower = flatter background)",
                        "non-ET bias (closer to 0 = no global offset)"]):
    for k, s in enumerate(SCHEMES):
        v = [r[s][key] for r in recs]
        a.scatter(np.full(len(v), k) + rng.normal(0, 0.06, len(v)), v, s=12, alpha=0.6)
        a.scatter([k], [np.median(v)], marker="_", s=600, c="k", zorder=3)
    a.set_xticks(xs); a.set_xticklabels(SCHEMES, rotation=20)
    a.set_title(ttl, fontsize=9); a.grid(alpha=0.3)
    if key == "bias":
        a.axhline(0, color="k", lw=0.8)
plt.tight_layout(); plt.show()

# the correction should matter most when there is more tail to step over
fig, a = plt.subplots(figsize=(5.5, 3.8))
a.scatter([r["f_et"] for r in recs],
          [r["proposed"]["separation"] - r["p99"]["separation"] for r in recs], s=18)
a.axhline(0, color="k", lw=0.8)
a.set_xlabel(r"$f_{ET}$ (ET fraction of brain)")
a.set_ylabel("separation gain\n(proposed - p99)")
a.set_title("the correction should help most when the tail is biggest", fontsize=9)
a.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Caveats

* **$f_{ET}$ is measured on the whole volume**, and the percentile it corrects is also volume-level,
  so the two are consistent. If you switch to per-slice normalization, recompute $f_{ET}$ per slice
  too -- it swings by an order of magnitude between slices.
* **The rule assumes the ET voxels are exactly the top $f_{ET}$ of the distribution.** They are not:
  some enhancement is dimmer than the brightest healthy tissue (vessels, fat), and necrotic core
  inside the tumor is dark. `oracle` is immune to this and `proposed` is not, which is why the
  proposed/oracle divisor comparison is the honest measure of the approximation -- more so than the
  separation numbers, which can improve for the wrong reason.
* **ET is one label.** `cmap.yaml` sets `et_labels: [4]`; whole-tumor or edema would give a very
  different $f_{ET}$.
* **This needs a mask at normalization time, which you do not have at inference.** Subjects without a
  segmentation are skipped here. If the scheme works, the next question is a mask-free stand-in --
  e.g. a fixed percentile offset fit to these results, or estimating the tail fraction from the
  T1ce/T1 histogram divergence directly.